# Extracting the netlist from the GDS

In [ ]:
# See Jane street's challenge https://blog.janestreet.com/can-you-reverse-engineer-an-asic/
!git clone https://github.com/janestreet/asic-puzzle-2026.git

fatal: destination path 'asic-puzzle-2026' already exists and is not an empty directory.


In [ ]:
# Setup and install conda
# Install condacolab to handle EDA tool dependencies efficiently
!pip install -q condacolab
import condacolab
condacolab.install()

# NOTE: Colab will restart its runtime after running condacolab.install(). This is normal. Continue to the next cell once it finishes

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:14
🔁 Restarting kernel...


In [ ]:
# 1. Install Magic from Litex-hub via Conda
!conda install -c litex-hub -c conda-forge magic -y

# 2. Install Volare via Pip (PyPI)
!pip install volare

# # 3. Fetch and enable Sky130A
# !volare fetch --pdk sky130 sky130A # wont work, Volare tracks PDK builds by their open_pdks commit hash rather than the broad string "sky130A"

# Fetch and enable a known stable sky130 build hash
# (This hash corresponds to a production-tested Sky130A open_pdks release)
!volare enable --pdk sky130 44a43c23c81b45b8e774ae7a84899a5a778b6b0b

Channels:
 - litex-hub
 - conda-forge
Platform: linux-64
Solving environment: - \ done

# All requested packages already installed.

no oauth token found for github.com
Version 44a43c23c81b45b8e774ae7a84899a5a778b6b0b not found locally, attempting 
to download…
⠙ Unpacking common.tar.zst…
⠴ Unpacking sky130_fd_io.tar.zst…
⠋ Unpacking sky130_fd_pr.tar.zst…
⠙ Unpacking sky130_fd_sc_hd.tar.zst…
⠸ Unpacking sky130_fd_sc_hvl.tar.zst…
⠋ Unpacking sky130_ml_xx_hd.tar.zst…
⠦ Unpacking sky130_sram_macros.tar.zst…
⠋ Enabling version 44a43c23c81b45b8e774ae7a84899a5a778b6b0b…
Version 44a43c23c81b45b8e774ae7a84899a5a778b6b0b enabled for the sky130 PDK.


In [ ]:
import os
tech_path = os.path.expanduser("~/.volare/sky130A/libs.tech/magic/sky130A.tech")
print("Tech file exists:", os.path.exists(tech_path))

Tech file exists: True


# Run Extraction with Magic

In [ ]:
import os
import subprocess

def extract_gds_to_spice(gds_path, top_cell_name, output_spice="extracted_netlist.spice", blackbox_stdcells=True):
    # Locate Sky130A tech file from volare installation
    pdk_root = os.path.expanduser("~/.volare")
    tech_file = os.path.join(pdk_root, "sky130A/libs.tech/magic/sky130A.tech")

    if not os.path.exists(tech_file):
        raise FileNotFoundError(f"Sky130 tech file not found at {tech_file}. Check Volare installation.")

    # Generate Magic Tcl script for non-interactive extraction
    tcl_commands = f"""
tech load {tech_file}
gds read {gds_path}
load {top_cell_name}
select top cell

# Extraction settings
extract do local
extract all

# Configure ext2spice
ext2spice lvs
"""

    if blackbox_stdcells:
        # Treats subcircuits/standard cells as abstract blocks rather than transistor-level
        tcl_commands += "\next2spice subcircuit on\n"

    tcl_commands += f"""
ext2spice -o {output_spice}
exit
"""

    # Write Tcl commands to file
    tcl_filename = "run_extract.tcl"
    with open(tcl_filename, "w") as f:
        f.write(tcl_commands)

    print(f"Running Magic extraction on {gds_path} ({top_cell_name})...")

    # Run Magic in batch mode (-dnull for no display/GUI)
    cmd = ["magic", "-dnull", "-noconsole", "-rcfile", f"{pdk_root}/sky130A/libs.tech/magic/sky130A.magicrc", tcl_filename]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if os.path.exists(output_spice):
        print(f"Extraction successful! Extracted SPICE saved to: {output_spice}")
    else:
        print("Extraction failed or log printed warnings. Magic stdout/stderr log:")
        print(result.stdout)
        print(result.stderr)

print("Extractor helper loaded successfully.")

Extractor helper loaded successfully.


In [ ]:
GDS_FILE = '/content/asic-puzzle-2026/puzzle.gds'

In [ ]:
# Update these parameters with your file name and top cell
# GDS_FILE = "your_design.gds"
TOP_CELL = "puzzle"
OUTPUT_SPICE = "my_extracted_netlist.spice"

# Set blackbox_stdcells=True for gate-level subcircuit extraction,
# or blackbox_stdcells=False for full device/transistor-level extraction.
extract_gds_to_spice(
    gds_path=GDS_FILE,
    top_cell_name=TOP_CELL,
    output_spice=OUTPUT_SPICE,
    blackbox_stdcells=True
)

Running Magic extraction on /content/asic-puzzle-2026/puzzle.gds (puzzle)...
Extraction successful! Extracted SPICE saved to: my_extracted_netlist.spice


In [ ]:
!cat my_extracted_netlist.spice

* NGSPICE file created from puzzle.ext - technology: sky130A

.subckt sky130_fd_sc_hd__nor4_2 C D Y A B VGND VPWR VPB VNB
X0 a_281_297# B a_27_297# VPB sky130_fd_pr__pfet_01v8_hvt ad=0.135 pd=1.27 as=0.135 ps=1.27 w=1 l=0.15
X1 Y B VGND VNB sky130_fd_pr__nfet_01v8 ad=0.08775 pd=0.92 as=0.08775 ps=0.92 w=0.65 l=0.15
X2 a_27_297# A VPWR VPB sky130_fd_pr__pfet_01v8_hvt ad=0.135 pd=1.27 as=0.135 ps=1.27 w=1 l=0.15
X3 a_475_297# D Y VPB sky130_fd_pr__pfet_01v8_hvt ad=0.26 pd=2.52 as=0.135 ps=1.27 w=1 l=0.15
X4 VGND C Y VNB sky130_fd_pr__nfet_01v8 ad=0.08775 pd=0.92 as=0.08775 ps=0.92 w=0.65 l=0.15
X5 Y D a_475_297# VPB sky130_fd_pr__pfet_01v8_hvt ad=0.135 pd=1.27 as=0.135 ps=1.27 w=1 l=0.15
X6 VGND D Y VNB sky130_fd_pr__nfet_01v8 ad=0.169 pd=1.82 as=0.08775 ps=0.92 w=0.65 l=0.15
X7 VGND A Y VNB sky130_fd_pr__nfet_01v8 ad=0.08775 pd=0.92 as=0.08775 ps=0.92 w=0.65 l=0.15
X8 Y A VGND VNB sky130_fd_pr__nfet_01v8 ad=0.08775 pd=0.92 as=0.182 ps=1.86 w=0.65 l=0.15
X9 VGND B Y VNB sky130_fd_pr__nfe